# S1 — Validate inputs and aggregate coverage

This supporting Notebook checks synthetic public-input contracts, row-linkage counts, anonymized coverage, and expected output schemas. It never prints review text, names, identifiers, DOI, ORCID, or provider responses; it does not fit research models. Run once after 06 (`NC_VALIDATION_FINAL=0`) and again after 10 (`NC_VALIDATION_FINAL=1`). All reports are under `results/demo/validation`.

Synthetic demonstration results are for software validation only and have no scientific interpretation.

SOURCE_LINEAGE_LITERAL: `aspect_extraction_stats.ipynb`; `count_papers_reviews_distribution论文及评审数统计.ipynb`; anonymous QC from `主表及分析表构建.ipynb` and `分析.ipynb`. No source records or identity-bearing diagnostics are copied.


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "project_paths.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "project_paths.py").is_file():
    raise FileNotFoundError("Run from the release root or notebooks directory")
sys.path.insert(0, str(ROOT))
from src.project_paths import project_path

with (ROOT / "config" / "config.example.yaml").open(encoding="utf-8") as stream:
    CONFIG = yaml.safe_load(stream)
assert CONFIG["analysis"]["formal_country_min_papers"] == 500
assert CONFIG["analysis"]["large_team_cutoff"] == 50
assert CONFIG["genderize"]["confident_probability_threshold"] == 0.80
assert CONFIG["genderize"]["confident_count_threshold"] == 20
SAMPLE = project_path(CONFIG["paths"]["sample_data"])
WORK = project_path(CONFIG["paths"]["work_data"])
RESULTS = project_path("results/demo")
OUT = RESULTS / "validation"
OUT.mkdir(parents=True, exist_ok=True)
FINAL = os.getenv("NC_VALIDATION_FINAL", "0") == "1"
ASPECT_CLASSES = [
    "Methodology", "Results", "Experiments and Evaluation", "Data and Datasets",
    "Figures and Tables", "Writing, Language and Terminology", "Theory and Models",
    "Comparison and Context", "Paper Assessment", "Novelty and Impact",
    "Validity, Rigor and Quality", "Limitations",
]
checks = []
metrics = []

def check(name, condition, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(condition) else "FAIL", "detail": detail})

def metric(name, value):
    metrics.append({"metric": name, "value": value})

def required_table(path, columns):
    check("exists:" + path.name, path.is_file())
    if not path.is_file():
        return pd.DataFrame(columns=columns)
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    check("schema:" + path.name, set(columns).issubset(frame.columns))
    return frame

papers = required_table(SAMPLE / "sample_papers.csv", ["paper_code", "publication_year", "subject_code"])
reviews_path = SAMPLE / "sample_reviews.jsonl"
check("exists:sample_reviews.jsonl", reviews_path.is_file())
reviews = pd.read_json(reviews_path, lines=True) if reviews_path.is_file() else pd.DataFrame()
check("papers_nonempty", len(papers) > 0)
check("reviews_nonempty", len(reviews) > 0)
check("review_linkage", not reviews.empty and reviews["paper_code"].isin(papers["paper_code"]).all())
check("paper_key_unique", papers["paper_code"].is_unique)
check("review_key_unique", reviews["review_id"].is_unique)
metric("sample_papers", len(papers))
metric("sample_reviews", len(reviews))

sentences = sorted((WORK / "sentence_parts").glob("part-*.parquet"))
sentence_count = sum(len(pd.read_parquet(path, columns=["review_id"])) for path in sentences)
check("sentences_nonempty", sentence_count > 0)
metric("sentences", sentence_count)
mock_path = SAMPLE / "mock_aspect_responses.jsonl"
mock_count = sum(1 for line in mock_path.open(encoding="utf-8") if line.strip()) if mock_path.is_file() else 0
check("mock_responses_nonempty", mock_count > 0)
metric("mock_responses", mock_count)
term_counts = required_table(WORK / "review_term_counts.parquet", ["review_id", "term_raw", "count_in_review"])
normalized = required_table(WORK / "review_term_counts_normalized.parquet", ["review_id", "term_norm", "major_class_final"])
check("term_counts_nonempty", len(term_counts) > 0)
check("normalized_terms_nonempty", len(normalized) > 0)
classes = set(normalized["major_class_final"].dropna().astype(str))
check("all_12_classes", set(ASPECT_CLASSES).issubset(classes))
metric("review_term_rows", len(term_counts))
metric("mapped_class_count", len(classes.intersection(ASPECT_CLASSES)))
metric("taxonomy_row_coverage", float(normalized["major_class_final"].notna().mean()) if len(normalized) else 0)

author = required_table(WORK / "paper_authors.parquet", ["paper_code", "author_order", "gender_status", "author_affil_country_primary"])
background = required_table(WORK / "paper_background_features_v2.parquet", ["paper_code", "n_authors", "large_team_ge_50", "is_international_collaboration"])
review_v2 = required_table(WORK / "review_aspect_class_counts_v2.parquet", ["review_id", "paper_code", "aspect_total_count"] + ASPECT_CLASSES)
jsd_v2 = required_table(WORK / "paper_aspect_disagreement_v2.parquet", ["paper_code", "n_reviews_valid_for_jsd", "mean_pairwise_jsd"])
statuses = set(author["gender_status"].dropna().astype(str))
for status in ("confident", "uncertain", "unknown"):
    check("author_status:" + status, status in statuses)
gender = set(author.loc[author["gender_status"].eq("confident"), "name_inferred_gender"].dropna().astype(str))
check("confident_male", "male" in gender)
check("confident_female", "female" in gender)
check("country_missing", author["author_affil_country_primary"].fillna("").astype(str).str.strip().eq("").any())
check("international_collaboration", background["is_international_collaboration"].eq(1).any())
for size, expected in ((49, 0), (50, 1), (51, 1)):
    row = background.loc[background["n_authors"].eq(size)]
    check(f"background_flag_{size}", len(row) > 0 and row["large_team_ge_50"].eq(expected).all())
check("valid_jsd_review_pairs", jsd_v2["n_reviews_valid_for_jsd"].ge(2).any())
check("nonzero_jsd", pd.to_numeric(jsd_v2["mean_pairwise_jsd"], errors="coerce").gt(0).any())
check("US_reference_present", background["first_author_country"].eq("US").any())
check("male_reference_present", background["first_author_gender_confident"].eq("male").any())
for control in ("publication_year", "subject_codes_final", "n_authors"):
    check("control_varies:" + control, background[control].nunique(dropna=True) > 1)
metric("author_rows", len(author))
metric("valid_jsd_papers", int(jsd_v2["mean_pairwise_jsd"].notna().sum()))
metric("positive_aspect_reviews", int(review_v2["aspect_total_count"].gt(0).sum()))

if FINAL:
    strict_path = RESULTS / "strict" / "samples" / "main_strict_paper_jsd_no_large50.parquet"
    strict = required_table(strict_path, ["paper_code", "n_authors", "mean_pairwise_jsd"])
    for size, expected in ((49, True), (50, True), (51, False)):
        candidates = set(background.loc[background["n_authors"].eq(size), "paper_code"])
        retained = bool(candidates.intersection(set(strict["paper_code"])))
        check(f"strict_retention_{size}", bool(candidates) and retained == expected)
    check("strict_max_authors_50", not strict.empty and strict["n_authors"].le(50).all())
    expected_files = [
        RESULTS / "strict" / "tables" / "descriptive_main_sample_summary.csv",
        RESULTS / "robustness" / "first_batch" / "tables" / "R7_model_review_aspect_count_poisson_offset__country.csv",
        RESULTS / "robustness" / "second_batch" / "tables" / "ALL_second_batch_sample_counts.csv",
        RESULTS / "robustness" / "high_coverage" / "tables" / "descriptive_sample_summary_8_high_coverage.csv",
        RESULTS / "publication" / "model_paper_jsd_ols__country_gender_combined_main.csv",
        RESULTS / "figures" / "double_column" / "fig_aspect_deviation_combined_doublecolumn.png",
        RESULTS / "figures" / "single_column" / "fig_aspect_deviation_combined_rotated_v2.png",
    ]
    for path in expected_files:
        check("output:" + path.name, path.is_file())
    metric("strict_papers", len(strict))

pd.DataFrame(checks).to_csv(OUT / ("validation_checks_final.csv" if FINAL else "validation_checks_intermediate.csv"), index=False)
pd.DataFrame(metrics).to_csv(OUT / ("coverage_summary_final.csv" if FINAL else "coverage_summary_intermediate.csv"), index=False)
summary = {"mode": "final" if FINAL else "intermediate", "passed": sum(row["status"] == "PASS" for row in checks),
           "failed": sum(row["status"] == "FAIL" for row in checks), "synthetic_only": True}
(OUT / ("validation_status_final.json" if FINAL else "validation_status_intermediate.json")).write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary)
if summary["failed"]:
    raise AssertionError(f"S1 validation failed: {summary['failed']} checks; see aggregate CSV")
